In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data/train.csv')

In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
2110572 Take-Home Exam: CU Automatic Short Answer Scoring
Author: Your Name

Description:
  This script demonstrates a complete end-to-end approach for training
  and generating predictions in the short-answer scoring task. It:
    1) Reads in the training data (train.csv).
    2) Preprocesses text by combining "question" and "answer".
    3) Uses TF–IDF vectorization + a Random Forest Regressor as a simple baseline.
    4) Evaluates performance (e.g. via cross-validation MSE).
    5) Generates predictions for test.csv and writes submission.csv.

Notes:
  - This is a minimal baseline. You should iterate, engineer features, or
    use more advanced NLP approaches (fine-tune a Thai BERT, etc.).
  - We assume train.csv, test.csv, and sample_submission.csv are all in
    the same working directory as this script/notebook.
  - Do not use external data, but you may use pretrained language models.

Usage:
  Run in a Jupyter notebook or as a standalone Python script.
  In Jupyter, ensure you have the training and test files in your environment.
"""

import os
import numpy as np
import pandas as pd

# For text processing & modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor


def read_data():
    """
    Reads train.csv and test.csv, plus sample_submission.csv.
    Returns:
        train_df (pd.DataFrame): Contains columns [ID, set, question, answer, score].
        test_df (pd.DataFrame) : Contains columns [ID, set, question, answer].
        sub_df   (pd.DataFrame): Contains columns [ID, score] for the sample submission.
    """
    train_path = "data/train.csv"
    test_path = "data/test.csv"
    sample_sub_path = "data/sample_submission.csv"

    if not (os.path.exists(train_path) and
            os.path.exists(test_path)  and
            os.path.exists(sample_sub_path)):
        raise FileNotFoundError(
            "Make sure train.csv, test.csv, and sample_submission.csv "
            "are all in the current working directory."
        )

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sub_df = pd.read_csv(sample_sub_path)

    return train_df, test_df, sub_df


def combine_text(question, answer):
    """
    Combine question and answer text for a simple single-field representation.
    """
    if pd.isna(question):
        question = ""
    if pd.isna(answer):
        answer = ""
    # Simple approach: just join question + answer
    return question + " " + answer


def preprocess_data(train_df, test_df):
    """
    Preprocess the text data:
      - Combine question + answer
      - Return raw text list for train & test, plus the numeric target for train
    """
    # Combine question + answer into one text field
    train_df["text"] = train_df.apply(
        lambda row: combine_text(row["question"], row["answer"]), axis=1
    )
    test_df["text"] = test_df.apply(
        lambda row: combine_text(row["question"], row["answer"]), axis=1
    )

    # Extract the text fields
    X_train_texts = train_df["text"].tolist()
    X_test_texts = test_df["text"].tolist()

    # Score is the numeric target
    y_train = train_df["score"].values

    return X_train_texts, y_train, X_test_texts


def build_model():
    """
    Build and return a pipeline: TF-IDF + RandomForestRegressor.
    You can replace or enhance this with your own approach (e.g. neural nets).
    """
    # We'll do TF–IDF vectorization first, then random forest for regression
    # (You can add min_df, ngram_range, etc. to TfidfVectorizer to tune.)
    vectorizer = TfidfVectorizer()
    regressor = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    # We won't use a formal sklearn Pipeline here,
    # but we will replicate that pattern manually for clarity.
    return vectorizer, regressor


def run_cross_val(vectorizer, regressor, X_train_texts, y_train):
    """
    Example: run cross-validation on the combined approach
    to get an estimate of MSE.
    """
    # We must fit_transform the vectorizer on training data inside each fold.
    # Normally you'd do: Pipeline([...]) and cross_val_score that pipeline.
    # Here we do manual cross-validation to illustrate steps:

    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mses = []

    # Manual K-Fold
    for train_index, val_index in kf.split(X_train_texts):
        X_train_fold = [X_train_texts[i] for i in train_index]
        X_val_fold   = [X_train_texts[i] for i in val_index]
        y_train_fold = y_train[train_index]
        y_val_fold   = y_train[val_index]

        # Fit the vectorizer on the fold
        X_train_vec = vectorizer.fit_transform(X_train_fold)
        # Fit the regressor
        regressor.fit(X_train_vec, y_train_fold)

        # Transform the val fold
        X_val_vec = vectorizer.transform(X_val_fold)
        y_pred_val = regressor.predict(X_val_vec)

        fold_mse = mean_squared_error(y_val_fold, y_pred_val)
        mses.append(fold_mse)

    return mses


def train_final_model(vectorizer, regressor, X_train_texts, y_train):
    """
    Train the final model on the entire training set.
    Returns the trained vectorizer & regressor.
    """
    # Fit vectorizer on entire training set
    X_train_vec = vectorizer.fit_transform(X_train_texts)
    # Fit the regressor
    regressor.fit(X_train_vec, y_train)
    return vectorizer, regressor


def predict_test_and_save(vectorizer, regressor, X_test_texts, sub_df, output_name="submission.csv"):
    """
    Predict on test set, fill the 'score' column in sub_df, and save CSV.
    """
    X_test_vec = vectorizer.transform(X_test_texts)
    predictions = regressor.predict(X_test_vec)

    # Insert predictions into sub_df's 'score' column
    sub_df["score"] = predictions

    # Round or clip if desired, but official instructions just say "predict score"
    # If there's no constraint, we can keep them as floats, or you can do:
    # sub_df["score"] = np.round(predictions, 2)

    sub_df.to_csv(output_name, index=False)
    print(f"Submission file saved to: {output_name}")


def main():
    print("=== Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== Preprocessing data ===")
    X_train_texts, y_train, X_test_texts = preprocess_data(train_df, test_df)

    print("=== Building model ===")
    vectorizer, regressor = build_model()

    print("=== Cross-validation ===")
    # Run 5-fold CV for a quick measure of MSE
    cv_mses = run_cross_val(vectorizer, regressor, X_train_texts, y_train)
    print(f"CV MSEs: {cv_mses}")
    print(f"Mean CV MSE: {np.mean(cv_mses):.4f}")

    print("=== Training final model on all training data ===")
    # Build a fresh copy of the model for the final training
    vectorizer_final, regressor_final = build_model()
    vectorizer_final, regressor_final = train_final_model(
        vectorizer_final, regressor_final, X_train_texts, y_train
    )

    print("=== Predicting on test set ===")
    predict_test_and_save(vectorizer_final, regressor_final, X_test_texts, sub_df)

    print("All done! Check the submission.csv file.")


if __name__ == "__main__":
    main()


=== Reading data ===
=== Preprocessing data ===
=== Building model ===
=== Cross-validation ===
CV MSEs: [np.float64(2.0042766267123286), np.float64(2.941467294520548), np.float64(3.8449388888888887), np.float64(2.7379827256944447), np.float64(2.3982966145833333)]
Mean CV MSE: 2.7854
=== Training final model on all training data ===
=== Predicting on test set ===
Submission file saved to: submission.csv
All done! Check the submission.csv file.
